# tACS Bandit — Data Audit

What data exists, who is excluded and why, and how each derived measure is
computed. Section order follows the manuscript's Methods so the two can be read
side by side.

**Everything here is computed, not transcribed.** Section 8 emits the analytic
sample table for the manuscript. If a number in the Methods disagrees with a
number here, this notebook is right and the manuscript needs updating.

| Section | Methods section |
|---|---|
| 1. Sample and demographics | 2.1 Participants |
| 2. Task data and exclusions | 2.3 Task, 2.8 Exclusion criteria |
| 3. Behavioral dependent variables | 2.7 (moved up, next to its source data) |
| 4. Survey measures | 2.6 Measures |
| 5. Stimulation data | 2.2 Design, 2.4 tACS |
| 6. Stimulation-derived metrics | 2.5 EEG and E-field modeling |
| 7. Methods/code consistency | — (new) |
| 8. Analytic sample summary | Table 1 |

Expanded sample only. Run `results_paper.ipynb` with `SAMPLE='dissertation'`
to reproduce the defended analyses.

In [1]:
# ============================================================================
# 0. Setup
# ============================================================================

import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings('ignore')

from config import (
    SUBJECT_INFO, DISSERTATION_SUBJECTS, DATA_DIR, REPO_ROOT,
    EXCLUSION_THRESHOLDS, STIM_EXCLUSIONS, NO_EARCLIP_SUBJECTS,
    WIN_FRACTION, MIN_CLEAN_RUNS_PER_CONDITION,
)
from data_loading import load_all_subjects
from exclusions import apply_all_exclusions
from nic_files import discover_runs

SAMPLE = 'all'
DERIV = REPO_ROOT / 'derivatives'
MASTER_CSV = REPO_ROOT / 'data' / 'master_subject_data.csv'

subj = pd.read_csv(MASTER_CSV, dtype={'subject_id': str})
raw_trials = load_all_subjects(sample=SAMPLE, verbose=False)
excl = apply_all_exclusions(raw_trials, verbose=False)

trials = excl['data_clean']
run_excl = excl['run_exclusions']
h2_eligible = sorted(str(s) for s in excl['h2_eligible'])

# Collected as each section runs; section 8 turns it into the Methods table.
ANALYTIC_SAMPLES = []

def record(analysis, n, basis):
    ANALYTIC_SAMPLES.append({'Analysis': analysis, 'N': n, 'Basis for reduction': basis})

print(f'registry     : {len(SUBJECT_INFO)} subjects')
print(f'trial data   : {raw_trials["subject_id"].nunique()} subjects, {len(raw_trials)} trials')
print(f'after cleaning: {trials["subject_id"].nunique()} subjects, {len(trials)} trials')

registry     : 66 subjects
trial data   : 62 subjects, 36260 trials
after cleaning: 61 subjects, 33268 trials


## 1. Sample and demographics

*Methods §2.1.* Participants were recruited from an existing longitudinal
neuroimaging cohort (R01-AG067011). Registration in `config.SUBJECT_INFO` is
the definition of "enrolled" for these analyses — a subject not in that
dictionary is invisible to every downstream step, so the count below is the
ceiling on every N in this notebook.

In [2]:
# ============================================================================
# 1.1 Who is registered, and what raw data exists for them
# ============================================================================

import glob

rows = []
for sid in SUBJECT_INFO:
    beh_files = glob.glob(str(DATA_DIR / f'sub-{sid}' / '*.csv'))
    beh_runs = {int(Path(f).name.split('run-')[1][:2])
                for f in beh_files if 'run-' in Path(f).name}
    rows.append({
        'subject_id': sid,
        'behavioral_runs': len(beh_runs),
        'eeg_runs': len(discover_runs(sid)),
        'counterbalance': SUBJECT_INFO[sid].get('counterbalance', '?'),
        'earclip': SUBJECT_INFO[sid].get('earclip', True),
        'note': SUBJECT_INFO[sid].get('notes', ''),
    })
availability = pd.DataFrame(rows)

n_reg = len(availability)
n_beh = (availability['behavioral_runs'] > 0).sum()
n_eeg = (availability['eeg_runs'] > 0).sum()

print(f'Registered                : {n_reg}')
print(f'  with behavioral data    : {n_beh}')
print(f'  with EEG data           : {n_eeg}')
print(f'  with both               : {((availability["behavioral_runs"]>0) & (availability["eeg_runs"]>0)).sum()}')

no_beh = availability[availability['behavioral_runs'] == 0]
if len(no_beh):
    print(f'\nNo behavioral data ({len(no_beh)}):')
    for _, r in no_beh.iterrows():
        has_eeg = 'EEG session exists' if r['eeg_runs'] else 'no data at all'
        print(f'  sub-{r["subject_id"]}: {has_eeg}')
    print('  Subjects with an EEG session but no behavioral files were run; the')
    print('  task files did not reach the repository and are treated as lost.')

record('Registered', n_reg, '—')
record('With usable task data', n_beh,
       f'{n_reg - n_beh} without behavioral files')

Registered                : 66
  with behavioral data    : 62
  with EEG data           : 60
  with both               : 57

No behavioral data (4):
  sub-10961: EEG session exists
  sub-11066: no data at all
  sub-11472: EEG session exists
  sub-11439: EEG session exists
  Subjects with an EEG session but no behavioral files were run; the
  task files did not reach the repository and are treated as lost.


In [3]:
# ============================================================================
# 1.2 Demographics
# ============================================================================
# Reported for subjects contributing analyzable task data, which is the sample
# the manuscript describes.

analysis_subjects = sorted(trials['subject_id'].unique())
demo = subj[subj['subject_id'].isin(analysis_subjects)].copy()
demo['age'] = pd.to_numeric(demo['age'], errors='coerce')

age = demo['age'].dropna()
print(f'N = {len(demo)}')
print(f'Age: M = {age.mean():.1f}, SD = {age.std():.1f}, '
      f'range = {age.min():.1f}-{age.max():.1f}  (n = {len(age)})')

for col in ['gender', 'race', 'ethnicity']:
    if col not in demo.columns:
        continue
    counts = demo[col].value_counts(dropna=False)
    pct = 100 * counts / len(demo)
    print(f'\n{col.capitalize()} (missing {demo[col].isna().sum()}):')
    for level, n in counts.items():
        print(f'  {str(level)[:38]:40s} {n:3d}  ({pct[level]:.0f}%)')

edu = pd.to_numeric(demo['education_years'], errors='coerce').dropna()
print(f'\nEducation (years): M = {edu.mean():.1f}, SD = {edu.std():.1f}, '
      f'range = {edu.min():.0f}-{edu.max():.0f}  '
      f'(n = {len(edu)}, missing {len(demo) - len(edu)})')

N = 61
Age: M = 47.1, SD = 19.1, range = 22.5-79.2  (n = 60)

Gender (missing 0):
  Male                                      31  (51%)
  Female                                    30  (49%)

Race (missing 0):
  White                                     38  (62%)
  Black or African American                 12  (20%)
  Asian                                      8  (13%)
  Two or more races, or race not describ     3  (5%)

Ethnicity (missing 0):
  Not Hispanic or Latino                    53  (87%)
  Hispanic or Latino                         5  (8%)
  Ethnicity not described (Other)            3  (5%)

Education (years): M = 16.2, SD = 3.3, range = 2-26  (n = 58, missing 3)


In [4]:
# ============================================================================
# 1.3 Manuscript sentence, pre-filled
# ============================================================================

n_f = int((demo['gender'].astype(str).str.lower().str.startswith('f')).sum())
n_m = int((demo['gender'].astype(str).str.lower().str.startswith('m')).sum())

def pct_of(col, *labels):
    if col not in demo.columns:
        return np.nan
    s = demo[col].astype(str).str.lower()
    hit = s.apply(lambda v: any(l in v for l in labels))
    return 100 * hit.sum() / len(demo)

print('Paste into Methods §2.1 and check the wording:\n')
print(f'{len(demo)} adults ({n_f} female, {n_m} male; age M = {age.mean():.1f}, '
      f'SD = {age.std():.1f}, range = {age.min():.1f}-{age.max():.1f} years) '
      f'were recruited from an existing longitudinal neuroimaging cohort '
      f'(R01-AG067011; PI: Smith) at Temple University. The sample was '
      f'predominantly White ({pct_of("race", "white"):.0f}%), with Black or '
      f'African American ({pct_of("race", "black", "african"):.0f}%) and Asian '
      f'({pct_of("race", "asian"):.0f}%) participants also represented; '
      f'{pct_of("ethnicity", "not hispanic", "non-hispanic"):.0f}% identified '
      f'as non-Hispanic/Latino.')

Paste into Methods §2.1 and check the wording:

61 adults (30 female, 31 male; age M = 47.1, SD = 19.1, range = 22.5-79.2 years) were recruited from an existing longitudinal neuroimaging cohort (R01-AG067011; PI: Smith) at Temple University. The sample was predominantly White (62%), with Black or African American (20%) and Asian (13%) participants also represented; 87% identified as non-Hispanic/Latino.


## 2. Task data and exclusion criteria

*Methods §2.3, §2.8.* Eight runs of a probabilistic two-armed bandit, ~6 min
each. The good option pays out on 75% of trials and reverses every 25–29
trials.

Pre-registered exclusions ([osf.io/s9k64](https://osf.io/s9k64/overview)) are
applied **at the run level**, then aggregated to participants. Reporting both
levels matters: a participant can lose runs without being excluded, and the
Methods needs both counts.

| Criterion | Threshold |
|---|---|
| Missed trials | > 20% of run |
| Side bias | > 95% one spatial location |
| Stimulus bias | > 95% one stimulus |
| Rapid responding | median RT < 200 ms |
| Feedback invariance | 10+ consecutive losses without switching |

A separate registry (`config.STIM_EXCLUSIONS`) removes runs where the delivered
stimulation did not match the assigned condition. Those runs are excluded from
active-vs-sham comparisons but retained for sham-only analyses when the
behavior itself is valid.

In [5]:
# ============================================================================
# 2.1 Run-level exclusions
# ============================================================================

n_runs = len(run_excl)
flags = {
    'missed trials > 20%': 'flag_missed',
    'side bias > 95%': 'flag_side_bias',
    'stimulus bias > 95%': 'flag_stim_bias',
    'median RT < 200 ms': 'flag_rapid_rt',
    'feedback invariance': 'flag_feedback_invariance',
}

print(f'Total runs collected: {n_runs}\n')
print(f'{"criterion":26s} {"runs flagged":>13s}   {"% of runs":>9s}')
print('-' * 54)
for label, col in flags.items():
    if col not in run_excl.columns:
        continue
    n = int(run_excl[col].sum())
    print(f'{label:26s} {n:13d}   {100*n/n_runs:8.1f}%')

n_behav = int(run_excl['exclude_behavioral'].sum())
n_stim = int(run_excl['exclude_stim'].sum())
n_any = int((run_excl['exclude_behavioral'] | run_excl['exclude_stim']).sum())

print(f'\nRuns failing >=1 behavioral criterion : {n_behav} ({100*n_behav/n_runs:.0f}%)')
print(f'Runs excluded for stimulation error   : {n_stim}')
print(f'Unique runs excluded overall          : {n_any}')
print('\nSome runs meet more than one criterion, so the column above sums to')
print('more than the number of excluded runs.')

Total runs collected: 494

criterion                   runs flagged   % of runs
------------------------------------------------------
missed trials > 20%                    1        0.2%
side bias > 95%                       11        2.2%
stimulus bias > 95%                   25        5.1%
median RT < 200 ms                     1        0.2%
feedback invariance                    8        1.6%

Runs failing >=1 behavioral criterion : 41 (8%)
Runs excluded for stimulation error   : 3
Unique runs excluded overall          : 43

Some runs meet more than one criterion, so the column above sums to
more than the number of excluded runs.


In [6]:
# ============================================================================
# 2.2 Which runs, and for whom
# ============================================================================

bad = run_excl[run_excl['exclude_behavioral'] | run_excl['exclude_stim']].copy()
if len(bad):
    def reasons(row):
        out = [label for label, col in flags.items()
               if col in row.index and row[col]]
        if row.get('exclude_stim'):
            out.append('stimulation error')
        return ', '.join(out)
    bad['reason'] = bad.apply(reasons, axis=1)
    print(bad[['subject_id', 'run', 'condition', 'reason']]
          .sort_values(['subject_id', 'run']).to_string(index=False))

per_subject = (run_excl.assign(excluded=run_excl['exclude_behavioral'] |
                                        run_excl['exclude_stim'])
                       .groupby('subject_id')['excluded'].agg(['sum', 'count']))
per_subject.columns = ['runs_excluded', 'runs_total']
lost_all = per_subject[per_subject['runs_excluded'] == per_subject['runs_total']]
print(f'\nParticipants losing every run: {len(lost_all)}'
      f'{" -> " + ", ".join(lost_all.index) if len(lost_all) else ""}')

subject_id  run condition                                   reason
     10418    6    active                      feedback invariance
     10559    5  baseline                      stimulus bias > 95%
     10606    4      post                      stimulus bias > 95%
     10606    7      sham                      feedback invariance
     10716    5  baseline                      stimulus bias > 95%
     10716    6    active                          side bias > 95%
     10716    7    active                          side bias > 95%
     10716    8      post                          side bias > 95%
     10804    7    active                      feedback invariance
     10898    2    active                      stimulus bias > 95%
     10898    3    active                      stimulus bias > 95%
     10898    7      sham                      feedback invariance
     10998    6    active                        stimulation error
     11116    1  baseline                      stimulus bias >

In [7]:
# ============================================================================
# 2.3 Participant-level consequences
# ============================================================================

clean_subjects = set(trials['subject_id'].unique())
has_sham = {s for s, g in trials.groupby('subject_id') if (g['condition'] == 'sham').any()}
has_active = {s for s, g in trials.groupby('subject_id') if (g['condition'] == 'active').any()}

sham_runs = run_excl[(run_excl['condition'] == 'sham') &
                     ~run_excl['exclude_behavioral']]
h1_trials = trials[trials['condition'] == 'sham']

print(f'Participants with any clean run        : {len(clean_subjects)}')
print(f'  with >=1 clean sham run (H1)         : {len(has_sham)}')
print(f'  with >=1 clean active run            : {len(has_active)}')
print(f'  eligible for paired comparison (H2)  : {len(h2_eligible)}')
print(f'\nH1 sample: {len(has_sham)} participants, {len(sham_runs)} clean sham runs, '
      f'{len(h1_trials)} trials')
h2_trials = trials[trials['subject_id'].isin(h2_eligible) &
                   trials['condition'].isin(['sham', 'active'])]
print(f'H2 sample: {len(h2_eligible)} participants, {len(h2_trials)} trials')

dropped_h2 = sorted(clean_subjects - set(h2_eligible))
print(f'\nNot eligible for H2 ({len(dropped_h2)}): {", ".join(dropped_h2)}')
print(f'Requirement: at least {MIN_CLEAN_RUNS_PER_CONDITION} clean run in each condition.')

record('H1: baseline behavior (sham)', len(has_sham),
       f'{len(SUBJECT_INFO) - len(has_sham)} without a clean sham run')
record('H2: paired stimulation comparisons', len(h2_eligible),
       f'{len(clean_subjects) - len(h2_eligible)} without clean runs in both conditions')

Participants with any clean run        : 61
  with >=1 clean sham run (H1)         : 59
  with >=1 clean active run            : 58
  eligible for paired comparison (H2)  : 55

H1 sample: 59 participants, 116 clean sham runs, 8520 trials
H2 sample: 55 participants, 15789 trials

Not eligible for H2 (6): 10716, 10898, 11433, 11440, 11606, 11773
Requirement: at least 1 clean run in each condition.


## 3. Behavioral dependent variables

*Methods §2.7, moved up to sit beside the task data it derives from.*

Three families, each computed per participant per condition:

**Win-stay/lose-shift.** `p(stay|win)` is the proportion of trials repeating
the previous choice after reward; `p(shift|lose)` the proportion switching
after non-reward. The first trial of each run and any missed response are
excluded, since neither has a defined predecessor.

**Rescorla-Wagner.** Q(a,t+1) = Q(a,t) + α·[r(t) − Q(a,t)], choice by softmax
with inverse temperature β. Q initializes to 0.5 at the start of every run —
each run is a fresh bandit with new contingencies, so values must not carry
across. Three estimators are compared in §3.2.

**Reversal adaptation.** Trials-to-criterion is the number of trials after a
reversal until three consecutive correct choices, averaged over reversals
within a condition.

In [8]:
# ============================================================================
# 3.1 Win-stay / lose-shift
# ============================================================================

from wsls import compute_wsls_h1_h2

wsls_h1, wsls_h2 = compute_wsls_h1_h2(excl['data_h1'], excl['data_h2'])

print(f'H1 (sham only)      : {len(wsls_h1)} participants')
print(f'H2 (by condition)   : {wsls_h2["subject_id"].nunique()} participants, '
      f'{len(wsls_h2)} participant-conditions')
print()
print(wsls_h1[['p_stay_win', 'p_shift_lose', 'n_win_trials', 'n_lose_trials']]
      .describe().round(3).to_string())

thin = wsls_h1[(wsls_h1['n_win_trials'] < 20) | (wsls_h1['n_lose_trials'] < 20)]
if len(thin):
    print(f'\nParticipants with <20 trials in either cell ({len(thin)}) — their'
          f' rates rest on few observations:')
    print(thin[['subject_id', 'n_win_trials', 'n_lose_trials']].to_string(index=False))

H1 (sham only)      : 59 participants
H2 (by condition)   : 55 participants, 110 participant-conditions

       p_stay_win  p_shift_lose  n_win_trials  n_lose_trials
count      59.000        59.000        59.000         59.000
mean        0.841         0.560        80.271         59.390
std         0.163         0.182        12.154         10.299
min         0.423         0.200        41.000         25.000
25%         0.748         0.442        74.000         53.000
50%         0.910         0.522        80.000         60.000
75%         0.962         0.657        85.500         67.500
max         1.000         1.000       104.000         85.000


### 3.2 Rescorla-Wagner: three estimators

The same model fitted three ways. They differ only in what constrains the
parameter estimates, and the differences are largest exactly where the data are
thinnest — which is where it matters.

**Maximum likelihood (MLE).** Picks the α, β maximizing the likelihood of the
observed choices, with no constraint beyond the parameter bounds. Each
participant and condition is fitted in isolation. When a participant's choices
are consistent with a whole range of parameter values, nothing stops the
optimizer settling at the edge of that range, and α pinned at 0 or 1 is not an
estimate of anything.

**Maximum a posteriori (MAP).** Adds weakly informative priors — Beta(2,2) on
α, Gamma(2,5) on β — and maximizes the posterior instead. Beta(2,2) has zero
density at 0 and 1, so boundary estimates become impossible. This is what
Methods §2.7.2 describes.

**Hierarchical Bayesian.** Estimates all participants jointly, with
group-level distributions acting as priors learned from the data rather than
assumed. A participant with weak data is pulled toward the group; one with
strong data is barely moved. It also returns a full posterior per participant
rather than a point estimate, so uncertainty propagates into anything computed
downstream. This is the estimator the manuscript will report.

Shrinkage is the common thread. MLE applies none, MAP applies a fixed amount
set by the analyst, hierarchical applies an amount the data determine.

In [9]:
# ============================================================================
# 3.2a Fit MLE and MAP, and load the hierarchical fit
# ============================================================================

from rescorla_wagner import fit_rw_by_condition

rw_mle = fit_rw_by_condition(trials, method='mle', verbose=False)
rw_map = fit_rw_by_condition(trials, method='map', verbose=False)

hb_path = DERIV / 'rl_models' / 'rw_within_all_subjects.csv'
rw_hb = (pd.read_csv(hb_path, dtype={'subject_id': str})
         if hb_path.exists() else None)

print(f'MLE : {len(rw_mle)} participant-condition fits')
print(f'MAP : {len(rw_map)} participant-condition fits')
if rw_hb is not None:
    print(f'HB  : {len(rw_hb)} participants (both conditions jointly)')
else:
    print('HB  : not found — run `python -m rl_models.run_within_fit --sample all`')

MLE : 117 participant-condition fits
MAP : 117 participant-condition fits
HB  : 55 participants (both conditions jointly)


In [10]:
# ============================================================================
# 3.2b How much do they differ?
# ============================================================================

BOUND_LO, BOUND_HI = 0.005, 0.995

comp = rw_mle.merge(rw_map, on=['subject_id', 'condition'],
                    suffixes=('_mle', '_map'))

print(f'{"":10s} {"alpha at boundary":>18s} {"mean alpha":>11s} {"mean beta":>10s}')
print('-' * 54)
for cond in ['sham', 'active']:
    d = comp[comp['condition'] == cond]
    for method in ['mle', 'map']:
        a = d[f'alpha_{method}']
        n_bound = int((a <= BOUND_LO).sum() + (a >= BOUND_HI).sum())
        print(f'{cond[:6]:6s} {method.upper():4s} {n_bound:15d} '
              f'{a.mean():14.3f} {d[f"beta_{method}"].mean():10.3f}')

if rw_hb is not None and 'sham_alpha' in rw_hb.columns:
    for cond in ['sham', 'active']:
        col = f'{cond}_alpha'
        a = rw_hb[col]
        n_bound = int((a <= BOUND_LO).sum() + (a >= BOUND_HI).sum())
        print(f'{cond[:6]:6s} {"HB":4s} {n_bound:15d} {a.mean():14.3f} '
              f'{rw_hb[f"{cond}_beta"].mean():10.3f}')

print('\nMAP and hierarchical both eliminate boundary estimates; MLE does not.')
print('That is the property the manuscript relies on, not a difference in fit.')

            alpha at boundary  mean alpha  mean beta
------------------------------------------------------
sham   MLE               18          0.636      4.338
sham   MAP                0          0.628      3.447
active MLE               21          0.706      4.225
active MAP                0          0.656      3.488
sham   HB                 0          0.673      2.944
active HB                 0          0.701      2.760

MAP and hierarchical both eliminate boundary estimates; MLE does not.
That is the property the manuscript relies on, not a difference in fit.


In [11]:
# ============================================================================
# 3.2c Agreement between estimators
# ============================================================================
# Rank correlation as well as Pearson: beta is bounded only at 50 under MLE, so
# a couple of extreme values dominate the covariance while rank order is
# preserved. Reporting only Pearson would misrepresent the agreement.

def agreement(x, y, label):
    ok = x.notna() & y.notna()
    if ok.sum() < 5:
        print(f'  {label:28s} too few overlapping values')
        return
    r, _ = stats.pearsonr(x[ok], y[ok])
    rho, _ = stats.spearmanr(x[ok], y[ok])
    print(f'  {label:28s} pearson {r:+.3f}   spearman {rho:+.3f}   n = {ok.sum()}')

print('MLE vs MAP')
for cond in ['sham', 'active']:
    d = comp[comp['condition'] == cond]
    agreement(d['alpha_mle'], d['alpha_map'], f'{cond} alpha')
    agreement(d['beta_mle'], d['beta_map'], f'{cond} beta')

if rw_hb is not None:
    print('\nMLE vs hierarchical')
    for cond in ['sham', 'active']:
        d = (comp[comp['condition'] == cond]
             .merge(rw_hb, on='subject_id', how='inner'))
        if f'{cond}_alpha' in d.columns:
            agreement(d['alpha_mle'], d[f'{cond}_alpha'], f'{cond} alpha')
            agreement(d['beta_mle'], d[f'{cond}_beta'], f'{cond} beta')

MLE vs MAP
  sham alpha                   pearson +0.906   spearman +0.897   n = 59
  sham beta                    pearson +0.923   spearman +0.870   n = 59
  active alpha                 pearson +0.921   spearman +0.886   n = 58
  active beta                  pearson +0.832   spearman +0.811   n = 58

MLE vs hierarchical
  sham alpha                   pearson +0.954   spearman +0.916   n = 55
  sham beta                    pearson +0.580   spearman +0.768   n = 55
  active alpha                 pearson +0.807   spearman +0.780   n = 55
  active beta                  pearson +0.184   spearman +0.777   n = 55


In [12]:
# ============================================================================
# 3.3 Reversal adaptation
# ============================================================================

from reversal_analysis import identify_reversals, compute_trials_to_criterion

CRITERION = 3
rev = identify_reversals(trials.copy(), window_pre=5, window_post=15, verbose=False)
ttc = compute_trials_to_criterion(rev, criterion=CRITERION)

print(f'Reversals identified : {rev["reversal_id"].nunique()} across '
      f'{rev["subject_id"].nunique()} participants')
print(f'TTC computed for     : {len(ttc)} reversals, '
      f'{ttc["subject_id"].nunique()} participants')
print(f'Criterion            : {CRITERION} consecutive correct choices\n')
print(ttc.groupby('condition')['trials_to_criterion']
        .agg(['count', 'mean', 'std']).round(3).to_string())

reached = ttc['reached_criterion'].mean() if 'reached_criterion' in ttc else np.nan
if np.isfinite(reached):
    print(f'\nReversals where criterion was reached within the window: {100*reached:.0f}%')
    print('Reversals never reaching criterion contribute no TTC value, so')
    print('participant means are based on differing numbers of reversals.')

Reversals identified : 909 across 61 participants
TTC computed for     : 909 reversals, 61 participants
Criterion            : 3 consecutive correct choices

           count   mean    std
condition                     
active       193  3.990  3.503
baseline     185  3.741  3.292
post         191  3.843  3.228
sham         203  3.793  3.510

Reversals where criterion was reached within the window: 85%
Reversals never reaching criterion contribute no TTC value, so
participant means are based on differing numbers of reversals.


In [13]:
# ============================================================================
# 3.4 Coverage of the behavioral DVs
# ============================================================================

families = {
    'WSLS (sham)': ['sham_p_stay_win', 'sham_p_shift_lose'],
    'WSLS (active)': ['active_p_stay_win', 'active_p_shift_lose'],
    'R-W (sham)': ['sham_alpha', 'sham_beta'],
    'R-W (active)': ['active_alpha', 'active_beta'],
    'Accuracy': ['sham_accuracy', 'active_accuracy'],
}
n_total = len(SUBJECT_INFO)
for family, cols in families.items():
    present = [f'{c}={subj[c].notna().sum()}' for c in cols if c in subj.columns]
    print(f'  {family:18s} ' + '  '.join(present) + f'   (of {n_total} registered)')
print('\nMeasures within a family share an N by construction: sham measures')
print('cover everyone with a clean sham run, active measures everyone eligible')
print('for the paired comparison.')

  WSLS (sham)        sham_p_stay_win=59  sham_p_shift_lose=59   (of 66 registered)
  WSLS (active)      active_p_stay_win=55  active_p_shift_lose=55   (of 66 registered)
  R-W (sham)         sham_alpha=59  sham_beta=59   (of 66 registered)
  R-W (active)       active_alpha=55  active_beta=55   (of 66 registered)
  Accuracy           sham_accuracy=59  active_accuracy=55   (of 66 registered)

Measures within a family share an N by construction: sham measures
cover everyone with a clean sham run, active measures everyone eligible
for the paired comparison.


## 4. Survey and cognitive measures

*Methods §2.6.*

**Cognitive function (§2.6.1).** The parent protocol's full battery spans eight
measures across three domains. Three of them — Digit Span, BVMT, Trail Making A
— were administered only to participants aged 40+, creating missingness that is
structurally confounded with age. Since age is this study's primary moderator, a
composite built from whatever each participant happened to complete would
confound the moderator with composite composition.

The primary composite therefore uses the five measures with coverage across all
ages: HVLT-R Total Immediate Recall, Salthouse Letter, Salthouse Pattern, TabCAT
Flanker, TabCAT Running Dots. Z-scored and averaged directly, no intermediate
domain grouping.

**Reward/punishment sensitivity (§2.6.2).** SPSRQ-RC, 20 items, 5-point scale.
SR predicts baseline p(stay|win) and SP predicts p(shift|lose) under H1.2.

In [14]:
# ============================================================================
# 4.1 Cognitive battery coverage, and the age-dependence of missingness
# ============================================================================

reduced = ['hvlt_total', 'salthouse_letter', 'salthouse_pattern',
           'flanker_score', 'running_dots_score']
age_restricted = ['digit_span_total', 'bvmt_total', 'trails_a_time']

sub_age = pd.to_numeric(subj['age'], errors='coerce')
print(f'{"measure":24s} {"n":>4s} {"% present":>10s} {"mean age present":>17s}')
print('-' * 60)
for label, group in [('reduced composite', reduced), ('age-restricted', age_restricted)]:
    print(f'  [{label}]')
    for m in group:
        if m not in subj.columns:
            continue
        have = subj[m].notna()
        print(f'  {m:22s} {have.sum():4d} {100*have.mean():9.0f}% '
              f'{sub_age[have].mean():17.1f}')

print('\nThe age-restricted measures are present almost exclusively for older')
print('participants — that is the age-dependent missingness described above,')
print('and the reason the reduced composite is primary.')

measure                     n  % present  mean age present
------------------------------------------------------------
  [reduced composite]
  hvlt_total               66       100%              46.9
  salthouse_letter         65        98%              47.0
  salthouse_pattern        65        98%              47.0
  flanker_score            63        95%              46.2
  running_dots_score       57        86%              45.3
  [age-restricted]
  digit_span_total         26        39%              65.4
  bvmt_total               30        45%              65.7
  trails_a_time            28        42%              65.2

The age-restricted measures are present almost exclusively for older
participants — that is the age-dependent missingness described above,
and the reason the reduced composite is primary.


In [15]:
# ============================================================================
# 4.2 Composite construction and properties
# ============================================================================

comp_df = subj.copy()
for m in reduced:
    if m in comp_df.columns:
        v = pd.to_numeric(comp_df[m], errors='coerce')
        comp_df[f'{m}_z'] = (v - v.mean()) / v.std()

z_cols = [f'{m}_z' for m in reduced if f'{m}_z' in comp_df.columns]
comp_df['global_reduced'] = comp_df[z_cols].mean(axis=1)

n_all5 = comp_df[reduced].notna().all(axis=1).sum()
print(f'Participants with all 5 measures : {n_all5}')
print(f'Participants with >=4 of 5       : {(comp_df[reduced].notna().sum(axis=1) >= 4).sum()}')
print(f'Composite non-null               : {comp_df["global_reduced"].notna().sum()}')

# Internal consistency of the reduced composite.
z = comp_df[z_cols].dropna()
if len(z) > 3:
    k = z.shape[1]
    alpha_c = (k / (k - 1)) * (1 - z.var(axis=0, ddof=1).sum() /
                               z.sum(axis=1).var(ddof=1))
    print(f'\nCronbach alpha (n = {len(z)}): {alpha_c:.3f}')

d = comp_df[['global_reduced', 'age']].apply(pd.to_numeric, errors='coerce').dropna()
r_age, p_age = stats.pearsonr(d['age'], d['global_reduced'])
print(f'Composite x age        : r = {r_age:+.3f}, p = {p_age:.4f}, n = {len(d)}')
d2 = comp_df[['global_reduced', 'education_years']].apply(pd.to_numeric, errors='coerce').dropna()
r_ed, p_ed = stats.pearsonr(d2['education_years'], d2['global_reduced'])
print(f'Composite x education  : r = {r_ed:+.3f}, p = {p_ed:.4f}, n = {len(d2)}')
print(f'\nShared variance with age: {100*r_age**2:.0f}%. Age x Cognition')
print('interactions must be read with that overlap in mind.')

record('Cognition-dependent analyses', int(comp_df['global_reduced'].notna().sum()),
       f'{len(SUBJECT_INFO) - int(comp_df["global_reduced"].notna().sum())} without composite')

Participants with all 5 measures : 56
Participants with >=4 of 5       : 62
Composite non-null               : 66

Cronbach alpha (n = 56): 0.737
Composite x age        : r = -0.655, p = 0.0000, n = 65
Composite x education  : r = -0.006, p = 0.9637, n = 63

Shared variance with age: 43%. Age x Cognition
interactions must be read with that overlap in mind.


In [16]:
# ============================================================================
# 4.3 Questionnaire coverage
# ============================================================================

questionnaires = {
    'SPSRQ reward (SR)': 'spsrq_sr',
    'SPSRQ punishment (SP)': 'spsrq_sp',
    'Sleep quality (BPSQI)': 'bpsqi_global',
    'Mindfulness (FFMQ)': 'ffmq_total',
    'Cognitive reflection (CRT)': 'crt_total',
    'Body sensations (BBS)': 'bbs_avg',
    'Childhood trauma (CTQ)': 'ctq_total',
    'Alcohol use (AUDIT)': 'audit_total',
    'PROMIS anxiety': 'promis_anxiety',
    'Loneliness (UCLA-3)': 'loneliness_total',
    'Education (years)': 'education_years',
}
n_total = len(SUBJECT_INFO)
print(f'{"measure":30s} {"n":>4s} {"missing":>8s}')
print('-' * 46)
for label, col in questionnaires.items():
    if col not in subj.columns:
        print(f'  {label:28s} {"--":>4s}   column absent')
        continue
    n = int(subj[col].notna().sum())
    print(f'  {label:28s} {n:4d} {n_total-n:8d}')

record('SPSRQ analyses', int(subj['spsrq_sr'].notna().sum()),
       f'{n_total - int(subj["spsrq_sr"].notna().sum())} did not complete the questionnaire')
record('Education-dependent analyses', int(subj['education_years'].notna().sum()),
       f'{n_total - int(subj["education_years"].notna().sum())} without education data')

measure                           n  missing
----------------------------------------------
  SPSRQ reward (SR)              61        5
  SPSRQ punishment (SP)          61        5
  Sleep quality (BPSQI)          65        1
  Mindfulness (FFMQ)             65        1
  Cognitive reflection (CRT)     66        0
  Body sensations (BBS)          25       41
  Childhood trauma (CTQ)         66        0
  Alcohol use (AUDIT)            15       51
  PROMIS anxiety                 15       51
  Loneliness (UCLA-3)            13       53
  Education (years)              63        3


### 4.4 Note on questionnaire sources

Coverage here reflects several REDCap exports read together rather than any one
file. Successive exports of the same project are **not nested** — each release
has gained some fields and lost others, and in one case a calculated field
(AUDIT sum) had fewer values in a later export than an earlier one despite
living at the same event. Taking only the newest export silently loses data.

`cognitive_merge` therefore reads every available export and coalesces per
subject, newest first. The same applies to TabCAT, where the August export
covers five more participants but drops 36 columns.

Measures still at low coverage (AUDIT, PROMIS, loneliness) are limited by
collection, not extraction: those participants have the parent event row with
the fields blank.